**Lab type:** review  
**Course:** NL301 Natural Language Processing with Python  
**Lesson:** 10 — Evaluating NLP Models  
**Task:** Review NLP evaluation code and answer five questions about metric selection and common pitfalls.

## Setup

In [ ]:
!pip install nltk evaluate bert-score sacrebleu --quiet
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
import numpy as np


## The evaluation code to review

In [ ]:
# Summarisation output to evaluate
references = [
    "The model exceeded analyst expectations beating forecasts by a wide margin.",
    "Scientists discovered a new species of deep sea fish near the Pacific trench.",
]
hypotheses = [
    "The company beat forecasts significantly.",
    "Researchers found an unknown fish species in the Pacific Ocean.",
]

# Evaluation code under review — read before answering the questions below
scores = []
for ref, hyp in zip(references, hypotheses):
    score = sentence_bleu(ref, hyp)     # (A) reference is a string, not list of lists
    scores.append(score)

print("Per-sentence BLEU scores:", scores)
print("Mean BLEU:", np.mean(scores))


---
## Review Question 1: sentence_bleu requires list of lists

`sentence_bleu(reference_string, hypothesis)` passes a string as the reference. NLTK iterates over characters, not words — BLEU = 0.0 silently. Demonstrate and fix.

<details>
<summary>🔑 Reveal answer — Q1</summary>

**Why it silently fails:** `sentence_bleu(reference, hypothesis)` expects `reference` to be a list of reference translations, each itself a list of tokens — a list of lists. Passing a plain string causes NLTK to iterate over individual characters as tokens. The n-gram matches between character-level "tokens" and word-level hypothesis tokens are essentially zero, returning BLEU = 0.0 without raising an error.

**Correct usage:** `sentence_bleu([ref.split()], hyp.split(), smoothing_function=SmoothingFunction().method1)`. The outer list allows multiple reference translations (e.g., multiple valid translations of the same source sentence). `SmoothingFunction().method1` adds small counts for missing n-grams, preventing zero scores on short sentences where 4-gram overlap may be sparse.

</details>

In [ ]:
# Demonstrate the silent failure
ref_string = "The model exceeded analyst expectations beating forecasts"
hyp        = "The company beat forecasts significantly"

score_wrong = sentence_bleu(ref_string, hyp.split())
print(f"Wrong (string reference):  {score_wrong:.4f}")

# Fix: reference must be a list of tokenised references — list of lists
ref_correct = [ref_string.split()]        # [['The', 'model', ...]]
score_correct = sentence_bleu(ref_correct, hyp.split(),
                               smoothing_function=SmoothingFunction().method1)
print(f"Correct (list of lists):   {score_correct:.4f}")
# Explain why sentence_bleu expects list of lists (multiple reference translations allowed)


---
## Review Question 2: Macro F1 vs weighted F1 for rare entities

NER dataset: 3 PER, 1 ORG, 100 O tokens. Which metric surfaces performance on rare entity types?

<details>
<summary>🔑 Reveal answer — Q2</summary>

**Macro F1 surfaces rare entities:** With 100 O tokens, 3 PER, and 1 ORG, weighted F1 is dominated by the O class (which has near-perfect precision and recall). The weighted average barely moves even when ORG is entirely missed. Macro F1 treats each class equally — a fully missed ORG class contributes F1=0.0 with weight 1/3, pulling macro F1 down significantly.

**Best practice for NER:** Use macro F1 (or, better, the seqeval library which computes span-level F1 — entity spans must match exactly, not just individual tokens). For production monitoring, track per-entity-type F1 separately so rare types don't disappear into an aggregate number.

</details>

In [ ]:
from sklearn.metrics import f1_score, classification_report

# Simulated NER predictions (O = outside, PER = person, ORG = organisation)
true_labels = ['O']*95 + ['PER','PER','PER','ORG'] + ['O']*5
pred_labels = ['O']*95 + ['PER','PER','O',  'O']   + ['O']*5   # misses last PER and ORG

labels = ['O', 'PER', 'ORG']
print(classification_report(true_labels, pred_labels, labels=labels))
print("Macro F1:   ", f1_score(true_labels, pred_labels, labels=labels, average='macro'))
print("Weighted F1:", f1_score(true_labels, pred_labels, labels=labels, average='weighted'))
# Which metric better surfaces that ORG was entirely missed?


---
## Review Question 3: BLEU for paraphrase evaluation

BLEU uses n-gram overlap. For paraphrases where meaning is preserved but wording differs, BLEU = 0. When is BLEU inappropriate?

<details>
<summary>🔑 Reveal answer — Q3</summary>

**Why BLEU fails here:** BLEU counts exact n-gram overlap between hypothesis and reference. "The firm exceeded expectations by a wide margin" shares almost no n-grams with "The company beat forecasts significantly" despite identical meaning, producing BLEU ≈ 0. BLEU was designed for machine translation where surface wording closeness is a proxy for translation adequacy — it is inappropriate for paraphrase evaluation, abstractive summarisation, or any task where valid outputs are expected to differ in vocabulary.

**When to use BERTScore instead:** BERTScore computes cosine similarity between contextual token embeddings, capturing semantic equivalence regardless of surface form. It correlates better with human judgement on abstractive tasks. Use BLEU only when exact wording matters (e.g., technical documentation translation); prefer BERTScore or ROUGE-2 for tasks where paraphrase is acceptable or expected.

</details>

In [ ]:
from bert_score import score as bert_score

reference  = "The company beat forecasts significantly."
hypothesis = "The firm exceeded expectations by a wide margin."

# BLEU
bleu = sentence_bleu([reference.split()], hypothesis.split(),
                     smoothing_function=SmoothingFunction().method1)
print(f"BLEU:      {bleu:.4f}")

# BERTScore
P, R, F1 = bert_score([hypothesis], [reference], lang="en", verbose=False)
print(f"BERTScore F1: {F1[0].item():.4f}")
# Explain: when is lexical overlap (BLEU) insufficient for evaluation?


---
## Review Question 4: ROUGE variants for summarisation

ROUGE-1 vs ROUGE-2 vs ROUGE-L — which is standard for summarisation benchmarks and why?

<details>
<summary>🔑 Reveal answer — Q4</summary>

**ROUGE-1** counts unigram overlap — high recall for content words but insensitive to phrase structure and easily gamed by synonym substitution. **ROUGE-L** measures the longest common subsequence, capturing sentence-level fluency but less sensitive to content density. **ROUGE-2** counts bigram overlap, balancing content coverage (better than ROUGE-1) with sensitivity to phrase-level fidelity.

**Standard practice:** ROUGE-2 F1 is the most commonly reported metric on summarisation benchmarks (CNN/DailyMail, XSum) because it correlates better with human judgement than ROUGE-1 and is more interpretable than ROUGE-L. Most papers report ROUGE-1, ROUGE-2, and ROUGE-L together, but ROUGE-2 is the headline number used for model comparisons.

</details>

In [ ]:
# Install rouge-score if not available
try:
    from rouge_score import rouge_scorer
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'rouge-score', '--quiet'])
    from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

ref  = "Scientists discovered a new fish species in the deep Pacific Ocean."
hyp1 = "Researchers found an unknown species of fish near the Pacific trench."   # paraphrase
hyp2 = "New fish species was discovered by scientists in the Pacific."            # reordered

for hyp in [hyp1, hyp2]:
    scores = scorer.score(ref, hyp)
    print(f"Hypothesis: '{hyp[:60]}'")
    for k, v in scores.items():
        print(f"  {k}: P={v.precision:.3f} R={v.recall:.3f} F={v.fmeasure:.3f}")
    print()
# Which ROUGE variant do most summarisation papers report, and why?


---
## Review Question 5: Metric ranking for summarisation

Given a summarisation task, rank BLEU, ROUGE-2, and BERTScore for appropriateness. Justify each ranking.

<details>
<summary>🔑 Reveal answer — Q5</summary>

**Ranking for summarisation (most to least appropriate):**

1. **ROUGE-2** — the established standard for summarisation benchmarks; captures bigram content overlap, widely reported and comparable across papers, computationally trivial.

2. **BERTScore** — best for semantic faithfulness when paraphrase and abstraction are present; penalises hallucination better than ROUGE because it uses contextual embeddings, but scores are not always comparable across domains or model versions.

3. **BLEU** — designed for machine translation; penalises paraphrase and rewards verbose n-gram matches, so it systematically under-scores good abstractive summaries and is inappropriate as a primary summarisation metric.

</details>

In [ ]:
# Run all three metrics on the same summarisation example
ref_sum  = "The central bank raised interest rates to combat persistent inflation pressures."
hyp_sum  = "The bank increased rates to fight inflation."

# BLEU
bleu = sentence_bleu([ref_sum.split()], hyp_sum.split(),
                     smoothing_function=SmoothingFunction().method1)

# ROUGE-2
rouge2 = scorer.score(ref_sum, hyp_sum)['rouge2'].fmeasure

# BERTScore
P, R, F1 = bert_score([hyp_sum], [ref_sum], lang="en", verbose=False)

print(f"BLEU:         {bleu:.4f}")
print(f"ROUGE-2:      {rouge2:.4f}")
print(f"BERTScore F1: {F1[0].item():.4f}")
print()
print("Ranking for summarisation (most to least appropriate):")
print("1. ___ — because ___")
print("2. ___ — because ___")
print("3. ___ — because ___")
